# SOH model comparison on Colab

Run this notebook with `Runtime > Change runtime type > GPU`.

Expected project layout in Google Drive:

```text
MyDrive/UROP/
  soh_gru_dsconv_pipeline.py
  compare_soh_models.py
  raw_samples/
    *.pkl
```

The full comparison uses `20 x 300 x 3`, 50 epochs, and includes:
`persistence`, `flatten_mlp`, `curve_cnn`, `gru_only`, `gru_dsconv`.

In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

import torch

print('python:', sys.version)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda:', torch.version.cuda)
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU is not available. In Colab, choose Runtime > Change runtime type > GPU.')

python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.11.0+cu128
cuda available: True
cuda: 12.8
gpu: Tesla T4


## 1. Set data source and paths

Recommended: upload/copy the whole `UROP` folder to `MyDrive/UROP` first. If your path is different, edit `DRIVE_PROJECT_DIR`.

Alternative: set `DATA_SOURCE = 'zip_upload'` and upload a zip that contains `soh_gru_dsconv_pipeline.py`, `compare_soh_models.py`, and `raw_samples/`.

In [2]:
import zipfile

DATA_SOURCE = 'drive'  # 'drive' or 'zip_upload'

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/UROP')
ZIP_EXTRACT_DIR = Path('/content/UROP_from_zip')
WORK_DIR = Path('/content/UROP_run')

if DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = DRIVE_PROJECT_DIR
    OUTPUT_DIR = DRIVE_PROJECT_DIR / 'comparison_outputs_gpu'
elif DATA_SOURCE == 'zip_upload':
    from google.colab import files
    uploaded = files.upload()
    zip_paths = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
    if not zip_paths:
        raise FileNotFoundError('Upload a .zip file that contains the project files and raw_samples/.')
    if ZIP_EXTRACT_DIR.exists():
        shutil.rmtree(ZIP_EXTRACT_DIR)
    ZIP_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_paths[0], 'r') as zf:
        zf.extractall(ZIP_EXTRACT_DIR)
    candidates = [ZIP_EXTRACT_DIR] + [p for p in ZIP_EXTRACT_DIR.iterdir() if p.is_dir()]
    PROJECT_DIR = next(
        (p for p in candidates if (p / 'compare_soh_models.py').exists() and (p / 'raw_samples').exists()),
        ZIP_EXTRACT_DIR,
    )
    OUTPUT_DIR = Path('/content/comparison_outputs_gpu')
else:
    raise ValueError("DATA_SOURCE must be 'drive' or 'zip_upload'")

required = [
    PROJECT_DIR / 'soh_gru_dsconv_pipeline.py',
    PROJECT_DIR / 'compare_soh_models.py',
    PROJECT_DIR / 'raw_samples',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required files/folders: ' + ', '.join(missing))

print('Project:', PROJECT_DIR)
print('Output dir:', OUTPUT_DIR)

Mounted at /content/drive
Project: /content/drive/MyDrive/UROP
Output dir: /content/drive/MyDrive/UROP/comparison_outputs_gpu


## 2. Copy project to Colab local disk

Training from Drive can be slow. This copies scripts and `raw_samples` to `/content/UROP_run`, while saving outputs back to Drive.

In [3]:
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)

for name in ['soh_gru_dsconv_pipeline.py', 'compare_soh_models.py']:
    shutil.copy2(PROJECT_DIR / name, WORK_DIR / name)

shutil.copytree(PROJECT_DIR / 'raw_samples', WORK_DIR / 'raw_samples')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pkl_files = sorted((WORK_DIR / 'raw_samples').glob('*.pkl'))
print('work dir:', WORK_DIR)
print('pkl files:', len(pkl_files))
print('first files:', [p.name for p in pkl_files[:5]])

work dir: /content/UROP_run
pkl files: 18
first files: ['CALB_25_T25-1.pkl', 'CALCE_CX2_35.pkl', 'HNEI_18650_NMC_LCO_25C_0-100_0.5-1.5C_a.pkl', 'HUST_9-1.pkl', 'ISU-ILCC_G8C1.pkl']


## 3. Run full GPU comparison

If Colab runs out of memory, reduce `BATCH_SIZE` to `64`. If you only want to test the pipeline first, set `EPOCHS = 3` and `MODELS = 'persistence,gru_dsconv'`.

In [4]:
FIXED_LEN = 300
EARLY_CYCLE = 20
HORIZON = 0
BATCH_SIZE = 128
EPOCHS = 50
LR = 1e-3
WEIGHT_DECAY = 1e-5
SEED = 42
MODELS = 'all'

cmd = [
    sys.executable,
    '-u',
    'compare_soh_models.py',
    '--data-dir', 'raw_samples',
    '--output-dir', str(OUTPUT_DIR),
    '--fixed-len', str(FIXED_LEN),
    '--early-cycle', str(EARLY_CYCLE),
    '--horizon', str(HORIZON),
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--lr', str(LR),
    '--weight-decay', str(WEIGHT_DECAY),
    '--seed', str(SEED),
    '--models', MODELS,
    '--device', 'cuda',
]

print('Running:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=WORK_DIR, check=True)

Running:
/usr/bin/python3 -u compare_soh_models.py --data-dir raw_samples --output-dir /content/drive/MyDrive/UROP/comparison_outputs_gpu --fixed-len 300 --early-cycle 20 --horizon 0 --batch-size 128 --epochs 50 --lr 0.001 --weight-decay 1e-05 --seed 42 --models all --device cuda


CompletedProcess(args=['/usr/bin/python3', '-u', 'compare_soh_models.py', '--data-dir', 'raw_samples', '--output-dir', '/content/drive/MyDrive/UROP/comparison_outputs_gpu', '--fixed-len', '300', '--early-cycle', '20', '--horizon', '0', '--batch-size', '128', '--epochs', '50', '--lr', '0.001', '--weight-decay', '1e-05', '--seed', '42', '--models', 'all', '--device', 'cuda'], returncode=0)

## 4. Show results

In [7]:
import pandas as pd

metrics_path = OUTPUT_DIR / 'model_comparison_metrics.csv'
metrics = pd.read_csv(metrics_path)
display(metrics)

print('metrics:', metrics_path)
print('histories:', OUTPUT_DIR / 'histories')
print('predictions:', OUTPUT_DIR / 'predictions')

,model,RMSE,MAE,MAPE_percent,R2,EOL_Error_cycles
0,persistence,0.008584,0.000458,0.059623,0.987533,NaN
1,flatten_mlp,0.152982,0.128656,12.398875,-2.959191,NaN
2,curve_cnn,0.185696,0.159515,15.209371,-4.833514,NaN
3,gru_dsconv,0.288799,0.260322,25.214765,-13.109657,NaN
4,gru_only,0.430375,0.420700,41.130802,-30.334220,115.0


metrics: /content/drive/MyDrive/UROP/comparison_outputs_gpu/model_comparison_metrics.csv
histories: /content/drive/MyDrive/UROP/comparison_outputs_gpu/histories
predictions: /content/drive/MyDrive/UROP/comparison_outputs_gpu/predictions


## Optional: run only GRU + DSConv

Use this if you already ran other baselines or want to debug only the current model.

In [6]:
# Uncomment and run if needed.
# cmd = [
#     sys.executable, '-u', 'compare_soh_models.py',
#     '--data-dir', 'raw_samples',
#     '--output-dir', str(OUTPUT_DIR.parent / 'comparison_outputs_gpu_gru_dsconv_only'),
#     '--fixed-len', '300',
#     '--early-cycle', '20',
#     '--horizon', '0',
#     '--batch-size', '128',
#     '--epochs', '50',
#     '--models', 'gru_dsconv',
#     '--device', 'cuda',
# ]
# subprocess.run(cmd, cwd=WORK_DIR, check=True)